In [69]:
import numpy as np
import matplotlib.pyplot as plt
import catboost
import lightgbm
import pandas as pd
import xgboost
import tensorflow as tf


In [106]:
# load test.csv and train.csv as pandas dataframes
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [107]:
train.head(), train.shape, test.head(), test.shape

(   id        date country              store             product  num_sold
 0   0  2010-01-01  Canada  Discount Stickers   Holographic Goose       NaN
 1   1  2010-01-01  Canada  Discount Stickers              Kaggle     973.0
 2   2  2010-01-01  Canada  Discount Stickers        Kaggle Tiers     906.0
 3   3  2010-01-01  Canada  Discount Stickers            Kerneler     423.0
 4   4  2010-01-01  Canada  Discount Stickers  Kerneler Dark Mode     491.0,
 (230130, 6),
        id        date country              store             product
 0  230130  2017-01-01  Canada  Discount Stickers   Holographic Goose
 1  230131  2017-01-01  Canada  Discount Stickers              Kaggle
 2  230132  2017-01-01  Canada  Discount Stickers        Kaggle Tiers
 3  230133  2017-01-01  Canada  Discount Stickers            Kerneler
 4  230134  2017-01-01  Canada  Discount Stickers  Kerneler Dark Mode,
 (98550, 5))

In [108]:
# find all unique values in all columns and also the count of unique values
for col in train.columns:
    print(col, train[col].nunique())
    print(train[col].unique())
    print("dtype: ", train[col].dtype)
    print("")

id 230130
[     0      1      2 ... 230127 230128 230129]
dtype:  int64

date 2557
['2010-01-01' '2010-01-02' '2010-01-03' ... '2016-12-29' '2016-12-30'
 '2016-12-31']
dtype:  object

country 6
['Canada' 'Finland' 'Italy' 'Kenya' 'Norway' 'Singapore']
dtype:  object

store 3
['Discount Stickers' 'Stickers for Less' 'Premium Sticker Mart']
dtype:  object

product 5
['Holographic Goose' 'Kaggle' 'Kaggle Tiers' 'Kerneler'
 'Kerneler Dark Mode']
dtype:  object

num_sold 4037
[  nan  973.  906. ... 3446. 2266. 3996.]
dtype:  float64



In [109]:
#print how mnay have num_sold as 0 or nan
print(train[train['num_sold'].isnull() | (train['num_sold'] == 0)].shape)
#print how mnay have any column as nan or a string col as empty or id col as nan or date col as empty
print(train[train.isnull().any(axis=1) | (train['id'].isnull()) | (train['date'].str.len() == 0)].shape)

# show head of all records where num_sold is 0 or nan
train[train['num_sold'].isnull() | (train['num_sold'] == 0)].head()



(8871, 6)
(8871, 6)


,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
45,45,2010-01-01,Kenya,Discount Stickers,Holographic Goose,NaN
90,90,2010-01-02,Canada,Discount Stickers,Holographic Goose,NaN
135,135,2010-01-02,Kenya,Discount Stickers,Holographic Goose,NaN
180,180,2010-01-03,Canada,Discount Stickers,Holographic Goose,NaN


In [110]:
import pandas as pd
import holidays
from datetime import datetime

# Assuming train and country DataFrames are already defined
# train = pd.read_csv('path_to_train.csv')
# country = pd.read_csv('path_to_country.csv')
#parse the 'date' column as datetime 
train['date'] = pd.to_datetime(train['date'])
# Country mapping dictionary
country_mapping = {
    'Canada': 'CA',
    'Finland': 'FI',
    'Italy': 'IT',
    'Kenya': 'KE',
    'Norway': 'NO',
    'Singapore': 'SG'
}

def is_holiday(date, country_name):
    country_code = country_mapping.get(country_name)
    if country_code:
        country_holidays = holidays.CountryHoliday(country_code)
        return date in country_holidays
    return False

def get_season(date):
    month = date.month
    day = date.day
    if (month == 12 and day >= 21) or (month in [1, 2]) or (month == 3 and day < 20):
        return 'Winter'
    elif (month == 3 and day >= 20) or (month in [4, 5]) or (month == 6 and day < 21):
        return 'Spring'
    elif (month == 6 and day >= 21) or (month in [7, 8]) or (month == 9 and day < 22):
        return 'Summer'
    else:
        return 'Fall'

# Add is_holiday column to train DataFrame
train['is_holiday'] = train.apply(lambda row: is_holiday(row['date'], row['country']), axis=1)
# Add season column to train DataFrame
train['season'] = train['date'].apply(lambda x: get_season(pd.to_datetime(x)))
# Add saturday and sunday flags to train DataFrame
train['saturday'] = train['date'].apply(lambda x: x.weekday() == 5)
train['sunday'] = train['date'].apply(lambda x: x.weekday() == 6)


# print("how many have saturday as True: ", train[train['saturday'] == True].shape)
# print("how many have sunday as True: ", train[train['sunday'] == True].shape)
# print("unique values in season: ", train['season'].unique())
train.head()

,id,date,country,store,product,num_sold,is_holiday,season,saturday,sunday
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN,True,Winter,False,False
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0,True,Winter,False,False
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0,True,Winter,False,False
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0,True,Winter,False,False
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0,True,Winter,False,False


In [111]:
import pandas as pd

categorical_columns = ['country', 'store', 'product', 'is_holiday', 'season', 'saturday', 'sunday']
numerical_columns = ['num_sold']
# train = pd.get_dummies(train, columns=categorical_columns)
# Function to combine categorical columns and hash the result
num_buckets = 2000
def combined_hash(row):
    combined_string = ''.join([str(row[col]) for col in categorical_columns])
    x=  hash(combined_string)
    if(x<0):
        x%=(-num_buckets)
    else:
        x%=num_buckets
    return x 
    

# Apply the function to each row and create a new column 'hash_col'
print(train.shape)
train['hash_col'] = train.apply(combined_hash, axis=1)
print(train.shape)
print("unique values in hash_col: ", train['hash_col'].nunique())
print("range of values in hash_col: ", train['hash_col'].min(), train['hash_col'].max())
train.head()
#apply one hot encoding to the categorical columns
train = pd.get_dummies(train, columns=categorical_columns)
print(train.shape)

(230130, 10)
(230130, 11)
unique values in hash_col:  1507
range of values in hash_col:  -1999 1996
(230130, 28)


In [113]:
# aaply ordinal  transformation to the date column
train['date'] = pd.to_datetime(train['date'])
train['date'] = train['date'].apply(lambda x: x.toordinal())
print(train.shape)
train.head()

#apply sin cos transformation to the date column by using which date of the year out of 365 days can be represented as sin and cos
train['sin_date'] = np.sin(2 * np.pi * train['date'] / 365)
train['cos_date'] = np.cos(2 * np.pi * train['date'] / 365)
print(train.shape)
train.head()



(230130, 30)
(230130, 30)


,id,date,num_sold,hash_col,country_Canada,country_Finland,country_Italy,country_Kenya,country_Norway,country_Singapore,...,season_Fall,season_Spring,season_Summer,season_Winter,saturday_False,saturday_True,sunday_False,sunday_True,sin_date,cos_date
0,0,719163,NaN,-739,True,False,False,False,False,False,...,False,False,False,True,True,False,True,False,0.930724,-0.365723
1,1,719163,973.0,-1338,True,False,False,False,False,False,...,False,False,False,True,True,False,True,False,0.930724,-0.365723
2,2,719163,906.0,-281,True,False,False,False,False,False,...,False,False,False,True,True,False,True,False,0.930724,-0.365723
3,3,719163,423.0,981,True,False,False,False,False,False,...,False,False,False,True,True,False,True,False,0.930724,-0.365723
4,4,719163,491.0,-283,True,False,False,False,False,False,...,False,False,False,True,True,False,True,False,0.930724,-0.365723


In [114]:
#remove all records where num_sold is nan and store those nan records in a new dataframe called nan_df
nan_df = train[train['num_sold'].isnull()]
print("shape of nan_df: ", nan_df.shape)
#remove nan_df from train
print("shape of train before removing nan_df", train.shape)
train = train.drop(nan_df.index)
print("shape of train after removing nan_df", train.shape)

shape of nan_df:  (8871, 30)
shape of train before removing nan_df (230130, 30)
shape of train after removing nan_df (221259, 30)


In [115]:
nan_df.shape,nan_df.head()

((8871, 30),
       id    date  num_sold  hash_col  country_Canada  country_Finland  \
 0      0  719163       NaN      -739            True            False   
 45    45  719163       NaN       111           False            False   
 90    90  719163       NaN     -1651            True            False   
 135  135  719163       NaN      1747           False            False   
 180  180  719163       NaN      1610            True            False   
 
      country_Italy  country_Kenya  country_Norway  country_Singapore  ...  \
 0            False          False           False              False  ...   
 45           False           True           False              False  ...   
 90           False          False           False              False  ...   
 135          False           True           False              False  ...   
 180          False          False           False              False  ...   
 
      season_Fall  season_Spring  season_Summer  season_Winter  saturda

In [116]:
train.drop(columns=['date'], inplace=True)
